# Lecture: How a Neural Network Learns (MLP in NumPy)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain what a forward pass, a loss, and a gradient are in one sentence each.
- Describe the 4 stages of a training step: **Forward** (guess) → **Loss** (grade) → **Backward** (blame) → **Update** (adjust).
- Explain why layers need bends (ReLU) to learn curves, and how a prediction error becomes a weight update.
- Read the system map and narrate one full training step through every section (§§4–§11).


## The big idea

Training a neural network is like finding the lowest point of a valley while blindfolded. You feel the slope under your feet (that slope is the **gradient**), take one step downhill (scaled by the **learning rate**), and repeat. A neural network is just a recipe made of simple steps: multiply-and-add layers (called **Linear** layers) with a bend inserted between them (the bend is the **ReLU**, which simply replaces negative numbers with zero so the network can learn curves, not just straight lines). The **forward pass** runs the recipe and makes a prediction; the **loss** scores how wrong it was; **backpropagation** (nicknamed *backprop*) is the chain rule from calculus applied layer by layer, telling every weight how much it contributed to the mistake; then each weight takes one downhill step. Do that over small random chunks of data (**mini-batches**) again and again and the predictions get better — that is all `notebook.ipynb` does, written out by hand in NumPy with no helper library doing calculus for you. (The valley walk itself waits in Part 3 — first, meet the guess machinery.)


## Part 1 — The Forward Pass (Making a Guess)

Data → Linear → bend → scores → probabilities (§§4–7). Meet the guess machinery first; Part 2 grades it.

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. Reading order: **Part 1** = §§4–7 (the guess) → **Part 2** = §8 (the grade) → **Part 3** = §§9–§10 (blame → adjust) plus the §11 system map.

| Term | One-liner | Section |
|---|---|---|
| MLP | The whole mini-network: Linear → ReLU → Linear | §4 |
| Linear layer | Multiply-and-add: `X @ W + b` | §4 |
| ReLU | The bend: negatives become zero | §5 |
| Logits | Raw scores (any number, not probabilities yet) | §6 |
| Softmax | Scores → probabilities that sum to 1 | §7 |
| Cross-entropy loss | Wrongness score: −log(probability of the true class) | §8 |
| Gradient | The slope: which way is downhill, per weight | §9 |
| Backprop | Chain rule backwards through each layer's weights (× Wᵀ) | §9 |
| Learning rate | Step-size knob | §10 |
| Mini-batch / epoch / SGD | Data chunks / full passes / the update loop | §10 |


### §4 MLP and the Linear layer

**In one sentence: an MLP is just multiply-and-add layers with a bend between them, and a Linear layer is one multiply-and-add step `output = X @ W + b` where `W` (weights) and `b` (bias) are the numbers the network learns.**

The lab's network is Linear(2→3) → ReLU → Linear(3→2): two inputs in, two class scores out, one bend in the middle. Without the bend, stacked Linear layers collapse into a single Linear layer (two straight lines composed are still a straight line) — so the bend is what makes it a *neural* network instead of plain regression. `W` holds one weight per input→output connection; `b` shifts each output up or down. Read each output as a **detector** and each weight as a **dial**: dial up the inputs that matter, dial down the rest — the dot product is a weighted vote, i.e. how much the input looks like what this detector hunts. `b` is the detector's baseline threshold. Concrete shapes (memorize this reading): `X` = one student `[hours studied, practice tests]` → (1, 2); `W` = dials from 2 inputs to 3 detectors → (2, 3); out = 3 detector scores → (1, 3). Columns change 2 → 3 because we asked for 3 detectors. Watch it: the diagram shows the shapes, then tiny numbers go through one Linear step (plus each feature's vote).


In [ ]:
import numpy as np  # reuse numpy for the tiny linear math
import matplotlib.pyplot as plt  # reuse pyplot for the shape diagram
np.random.seed(2)  # fix the seed for reproducibility
X = np.array([[0.5, -1.0]])  # define one sample with 2 features (shape 1x2)
W = np.array([[1.0, 0.0, -2.0], [0.5, 1.5, 1.0]])  # define weights: 2 inputs -> 3 outputs (shape 2x3)
b = np.array([[0.1, -0.2, 0.0]])  # define one bias per output (shape 1x3)
out = X @ W + b  # THE linear layer: matrix multiply then add bias (shape 1x3)
print('X shape', X.shape, '-> out shape', out.shape)  # print shapes: columns change 2 -> 3
print('out:', np.round(out, 3))  # print values: [0.5*1 + -1*0.5 + 0.1, ...] = [0.1, -1.7, -2.0]
vote = X * W[:, 0:1].T  # each feature's vote for detector 0: [0.5*1.0, -1.0*0.5] (shape 1x2)
print('detector-0 votes:', np.round(vote[0], 3), '-> sum', round(float(vote.sum()), 3), '+ bias 0.1 =', round(float(out[0, 0]), 3))  # votes add, bias thresholds
fig, ax = plt.subplots(figsize=(7, 2.2))  # make a wide flat figure for the shape diagram
ax.axis('off')  # hide axes: this is a diagram, not a plot
ax.set_xlim(0, 10)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
boxes = [('x\n(1x2)', 0.5), ('Linear 2->3\nX@W+b', 3.5), ('ReLU\n(bend)', 6.0), ('Linear 3->2\nlogits', 8.5)]  # stage name + x-center
for name, xc in boxes:  # draw each stage box
    ax.text(xc, 1.5, name, ha='center', va='center', fontsize=9,  # centered stage label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box around it
for xa, xb in [(1.4, 2.6), (4.6, 5.4), (6.6, 7.4)]:  # arrow gaps between consecutive boxes
    ax.annotate('', xy=(xb, 1.5), xytext=(xa, 1.5),  # draw a forward arrow left -> right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.set_title('The MLP: data flows left to right through two Linear steps and one bend')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram


### §5 ReLU — the bend

**In one sentence: ReLU outputs `max(0, z)` — positives pass through, negatives become zero — and that single bend is what lets the network learn curves instead of only straight lines.**

Two facts matter later. Forward, it sparsifies: roughly half the neurons output exact zero on any input. Backward, it *switches the gradient off* where the input was ≤ 0 — nudging a dead neuron changes nothing, so the derivative is 0 there (1 elsewhere). Call this the **mask**; §9 uses it to stop blame at dead neurons. Watch it: the curve, plus the mask on five sample values.


In [ ]:
import numpy as np  # reuse numpy for the ReLU math
import matplotlib.pyplot as plt  # reuse pyplot for the bend plot
np.random.seed(3)  # fix the seed for reproducibility
z = np.linspace(-3, 3, 200)  # build 200 points across the bend for the curve
a = np.maximum(0, z)  # apply ReLU elementwise: negatives -> 0, positives unchanged
vals = np.array([-2.0, -0.5, 0.0, 1.0, 3.0])  # pick five sample pre-activation values
print('in :', vals)  # print the inputs going into ReLU
print('out:', np.maximum(0, vals))  # print the outputs: negatives flattened to 0
print('mask (derivative):', (vals > 0).astype(int))  # print the backward mask: 1 where alive, 0 where dead
fig, ax = plt.subplots(figsize=(5, 3.5))  # make a figure for the bend curve
ax.plot(z, a, label='ReLU max(0,z)')  # draw the bend: flat at 0, then 45-degree line
ax.plot(z, z, '--', alpha=0.4, label='identity (no bend)')  # dashed line: what a bend-less net would do
ax.set_xlabel('z (into ReLU)')  # label the input axis
ax.set_ylabel('a (out of ReLU)')  # label the output axis
ax.set_title('ReLU: flat left of 0 (dead zone), straight right of 0')  # title the bend story
ax.legend()  # show which line is which
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §6 Logits — raw scores, not probabilities yet

**In one sentence: logits are the final Linear layer's raw outputs — one score per class, any real number (negative is fine, bigger than 1 is fine) — and they only become probabilities after softmax (§7).**

Beginners often read logits as confidences ("score 2.0 means 200%?"). It doesn't — 2.0 vs 1.0 vs −1.5 only says *ranking*: class 0 is winning, class 2 is far behind. The absolute values are meaningless until normalized; even shifting all three by +100 changes nothing after softmax. Watch it: one sample's logits as bars, with the ± lines showing these are unconstrained numbers.


In [ ]:
import numpy as np  # reuse numpy (bars need no math, but keeps the habit)
import matplotlib.pyplot as plt  # reuse pyplot for the score bars
np.random.seed(4)  # fix the seed for reproducibility
logits = np.array([2.0, 1.0, -1.5])  # define raw scores for 3 classes (note: negative and >1 allowed)
print('logits:', logits)  # print the raw scores
print('sum:', logits.sum(), '(meaningless: not 1, can even be negative)')  # print why they are not probabilities
print('winner: class', int(np.argmax(logits)), '(ranking is all they tell us)')  # print the only valid reading
fig, ax = plt.subplots(figsize=(5, 3.5))  # make a figure for the score bars
ax.bar(['class 0', 'class 1', 'class 2'], logits, color='steelblue')  # draw one bar per raw score
ax.axhline(0, color='k', linewidth=0.8)  # draw the zero line for reference
ax.set_ylabel('logit (raw score, any value)')  # label: stress these are unconstrained
ax.set_title('Logits: ranking yes, probabilities no')  # title the takeaway
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §7 Softmax — scores become probabilities

**In one sentence: softmax exponentiates the logits and divides by the total, so the outputs are positive and sum to 1 — a probability distribution over the classes.**

Formula: `p[i] = exp(z[i] − max(z)) / Σ exp(z[j] − max(z))`. Subtracting the row max first changes nothing mathematically (it cancels in the division) but prevents `exp(1000) = inf` overflow — that one-line safety trick is in the lab's `softmax` and worth memorizing. exponentiation stretches gaps. Watch it: the same three scores before and after.


In [ ]:
import numpy as np  # reuse numpy for the softmax math
import matplotlib.pyplot as plt  # reuse pyplot for the before/after bars
np.random.seed(5)  # fix the seed for reproducibility
logits = np.array([[2.0, 1.0, -1.5]])  # reuse the §6 scores as one row (shape 1x3)
shifted = logits - logits.max(axis=1, keepdims=True)  # subtract row max:overflow-proofing vs exp overflow
exp = np.exp(shifted)  # exponentiate: all positive, gaps amplified
probs = exp / exp.sum(axis=1, keepdims=True)  # normalize: rows now sum to 1
print('probs:', np.round(probs[0], 4))  # print [0.66, 0.244, 0.02]-ish: winner amplified vs 2:1 logits
print('sum:', round(float(probs.sum()), 6))  # print the contract: exactly 1.0
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))  # make two side-by-side panels
ax[0].bar(['c0', 'c1', 'c2'], logits[0], color='steelblue')  # left: raw logits (can be negative)
ax[0].set_title('before: logits')  # left title
ax[0].set_ylabel('score')  # left label
ax[1].bar(['c0', 'c1', 'c2'], probs[0], color='darkorange')  # right: probabilities (0..1, sum 1)
ax[1].set_title('after: probabilities (sum = 1)')  # right title
ax[1].set_ylabel('probability')  # right label
plt.suptitle('Softmax turns ranking into a probability distribution')  # overall takeaway
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## Part 2 — The Loss (How Bad Was the Guess?)

One number grades the guess (§8: the model's surprise at the truth). Part 3 spends it as blame.

### §8 Cross-entropy loss — scoring wrongness with one number

**In one sentence: the loss is `-log(probability the model gave the TRUE class)` — near 0 when confident-and-right, huge when confident-and-wrong.**

Only the true class's probability matters; the other nine digits' scores are irrelevant to the number (they mattered only *through* softmax normalization). The log shape is the teacher: the difference between 0.9 and 0.99 (loss 0.105 → 0.010) rewards squeezing out the last doubt, while 0.1 vs 0.01 (2.30 → 4.61) punishes confident mistakes steeply. The lab averages this over the mini-batch (divides by N). Why `log`? Think **surprise**: an event with probability p carries −log(p) surprise, and independent surprises *add* (two fair coin tosses = twice one toss) — that additivity is what forces the log. A fair coin landing heads (p=0.5) is mildly surprising (−ln 0.5 ≈ 0.69); a 99%-sure thing happening is barely news (0.01). Cross-entropy is the model's surprise at the truth. Watch it: the curve, the coin landmarks, plus our two digit landmarks.


In [ ]:
import numpy as np  # reuse numpy for the log curve
import matplotlib.pyplot as plt  # reuse pyplot for the loss curve
np.random.seed(6)  # fix the seed for reproducibility
p = np.linspace(0.01, 1.0, 200)  # build 200 true-class probabilities from 1% to 100%
ce = -np.log(p)  # evaluate cross-entropy at each: -log(p)
print('confident+right p=0.9 -> loss', round(float(-np.log(0.9)), 3))  # print ~0.105: small
print('confident+WRONG p=0.1 -> loss', round(float(-np.log(0.1)), 3))  # print ~2.303: steep
print('fair coin p=0.5 -> surprise', round(float(-np.log(0.5)), 3))  # print 0.693: one coin-toss of surprise
print('near-certain p=0.99 -> surprise', round(float(-np.log(0.99)), 3))  # print 0.010: barely news
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the loss curve
ax.plot(p, ce, label='loss = -log(p_true)')  # draw the curve: flat near 1, steep near 0
ax.plot([0.9], [-np.log(0.9)], 'go', ms=8, label='right (p=0.9, loss 0.11)')  # mark the good point
ax.plot([0.1], [-np.log(0.1)], 'ro', ms=8, label='wrong (p=0.1, loss 2.30)')  # mark the bad point
ax.set_xlabel('probability given to the TRUE class')  # label: only this probability counts
ax.set_ylabel('loss (lower is better)')  # label: the number SGD minimizes
ax.set_title('Cross-entropy: gentle when right, brutal when confidently wrong')  # title the moral
ax.legend()  # show which dot is which
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## Part 3 — Backward + Update (Fixing the Dials)

Warm-up below: the blindfolded valley walk from the big idea, now precise in 1D (a single dial). Then §9 scales blame to matrices, §10 runs the loop, and §11 (below) maps the whole system.

In [ ]:
import numpy as np  # bring in numpy for vector math on small arrays
import matplotlib.pyplot as plt  # bring in pyplot for the loss-valley plot
np.random.seed(0)  # fix the seed so the plot is identical every run
w = 0.0  # ONE dial to tune (a net holds thousands inside matrices, §4); start far left of the min at w=3
lr = 0.3  # define learning rate: size of each downhill step (0.3 is stable here)
true_min = 3.0  # define the valley bottom: loss is zero exactly at w=3
traj = [w]  # create list holding every visited weight, starting with w=0
for step in range(12):  # repeat the gradient-descent update 12 times
    grad = 2.0 * (w - true_min)  # power rule: d/dw (w-3)^2 = 2(w-3) — the 2 is just the exponent coming down
    w = w - lr * grad  # take one SGD step: move against the gradient (downhill)
    traj.append(w)  # record the new weight so we can plot the path
traj = np.array(traj)  # convert the path list to a numpy array for easy math
xs = np.linspace(-1, 5, 200)  # build 200 x-points spanning the valley for the curve
ys = (xs - true_min) ** 2  # evaluate the loss L=(w-3)^2 at every x-point
losses = (traj - true_min) ** 2  # evaluate the loss at each visited weight
print('start loss:', round(float(losses[0]), 4))  # print loss at w=0 (should be 9.0)
print('final loss:', round(float(losses[-1]), 6))  # print loss after 12 steps (near zero)
print('final w:', round(float(traj[-1]), 4))  # print final weight (should be near 3.0)
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes for the plot
ax.plot(xs, ys, label='loss L=(w-3)^2')  # draw the valley curve itself
ax.plot(traj, losses, 'ro-', ms=4, label='downhill steps')  # draw red dots for each visited weight
ax.set_xlabel('weight w')  # label the x-axis (the thing we are tuning)
ax.set_ylabel('loss L')  # label the y-axis (how wrong we are)
ax.set_title('Gradient descent: 12 steps to the valley floor')  # title the story of the plot
ax.legend()  # show which line is which
plt.tight_layout()  # snug the layout so labels are not clipped
plt.show()  # render the figure


### §9 Gradient and backprop — finding blame, layer by layer

**In one sentence: the gradient is the slope of the loss with respect to each weight (sign = downhill direction), and backprop is the chain rule carrying the output error backwards through every layer so each weight learns its share of the blame.**

It starts with the **output error**: prediction minus answer (`ŷ − y` — the §7 probabilities with 1 subtracted at the true class, ÷ N for the batch mean). That vector *is* `dL/dlogits`, no Jacobian matrix needed. Then, layer by layer right-to-left: through a Linear layer, the chain rule multiplies the error by the weights transposed (`error · Wᵀ`) — each neuron collects blame from every output it fed, weighted by its dials; through ReLU, dead neurons switch their gradients off (§5's mask). The lab checks every gradient's shape matches its parameter — a free bug-catcher.

Why the transpose? A shape box (memorize this pattern): `A1` is (Batch, 3), `dlogits` is (Batch, 2), and `dW2` must match `W2`: (3, 2). So `dW2 = A1.T @ dlogits`, and (3 × Batch) @ (Batch × 2) = (3 × 2) ✓. In words: weight (i, j)'s blame is `Σ over batch of A1[b,i] × dlogits[b,j]` — its input times its output's error, summed over the batch — and that sum-of-products *is* a matrix multiply, hence the transpose. Watch it: slopes as tangent arrows, then the output-error numbers with their push-up/push-down reading.


In [ ]:
import numpy as np  # reuse numpy for slope + gradient math
import matplotlib.pyplot as plt  # reuse pyplot for the tangent plot
np.random.seed(7)  # fix the seed for reproducibility
xs = np.linspace(-1, 5, 200)  # rebuild the valley curve points
ys = (xs - 3.0) ** 2  # evaluate L=(w-3)^2 along it
for w0 in [0.0, 4.5]:  # check the slope at two spots: left of min, right of min
    print(f'slope at w={w0}: {2.0 * (w0 - 3.0):+.1f} -> downhill is', 'right' if w0 < 3 else 'left')  # sign = direction
logits = np.array([[2.0, 1.0, 0.1]])  # small 3-class example, true class 0
y = np.array([0])  # true label is class 0
shifted = logits - logits.max(axis=1, keepdims=True)  # stability shift (same trick as §7)
probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)  # softmax to probabilities
N = 1  # batch size 1 keeps the /N visible but trivial
dlogits = probs.copy()  # start from the prediction...
dlogits[0, y[0]] -= 1.0  # ...subtract the one-hot answer: (p - onehot)
dlogits = dlogits / N  # ...average over the batch: the full output error
print('dL/dlogits:', np.round(dlogits[0], 4))  # print: negative on truth (push up), positive elsewhere (push down)
onehot = np.zeros_like(probs)  # build the answer key: all zeros...
onehot[0, y[0]] = 1.0  # ...except 1 at the true class: [1, 0, 0]
err = probs - onehot  # output error = prediction minus truth (same numbers here: N=1)
print('one-hot truth:', onehot[0].astype(int))  # print the answer key
print('error vector:', np.round(err[0], 4))  # print the blame: truth too low (push UP), wrong too high (push DOWN)
fig2, ax2 = plt.subplots(figsize=(5, 2.6))  # second figure: the error vector as bars
ax2.bar(['c0 (truth)', 'c1', 'c2'], err[0], color='steelblue')  # one bar per class's signed error
ax2.axhline(0, color='k', linewidth=0.8)  # zero line: the sign tells the push direction
ax2.text(0, err[0, 0] - 0.04, 'push UP', ha='center', fontsize=9)  # truth scored low: push it up
ax2.text(1, err[0, 1] + 0.02, 'push DOWN', ha='center', fontsize=9)  # wrong scored high: push it down
ax2.set_ylabel('error (prediction - truth)')  # label: signed blame per class
ax2.set_title('Output error: negative = push up, positive = push down')  # title the reading rule
plt.tight_layout()  # snug the layout
plt.show()  # render the figure
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for slopes-as-arrows
ax.plot(xs, ys, label='loss valley')  # draw the valley
for w0, c in [(0.0, 'green'), (4.5, 'red')]:  # draw one tangent arrow per spot
    s = 2.0 * (w0 - 3.0)  # slope dL/dw at this weight
    ax.annotate('', xy=(w0 - np.sign(s) * 0.8, (w0 - np.sign(s) * 0.8 - 3.0) ** 2),  # arrowhead: one step downhill
                xytext=(w0, (w0 - 3.0) ** 2),  # arrow tail: current position
                arrowprops=dict(arrowstyle='->', color=c, lw=2))  # colored downhill arrow
    ax.plot(w0, (w0 - 3.0) ** 2, 'ko')  # mark the current position with a dot
ax.set_xlabel('weight w')  # label the tuned thing
ax.set_ylabel('loss L')  # label wrongness
ax.set_title('Gradient = slope; its sign points downhill (green: go right, red: go left)')  # title the rule
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §10 Learning rate, mini-batches, epochs, SGD — the loop's knobs

**In one sentence: the learning rate scales each downhill step (`w -= lr × grad`), a mini-batch is the small data chunk one step learns from, an epoch is one full pass over all chunks, and SGD is simply repeating batch-steps — total updates = epochs × (dataset ÷ batch).**

Learning rate is the only knob that can *break* training in one step: too small crawls (watch green below), just right converges (blue), too big overshoots and explodes (red — try `lr=1.0` in the lab to see it). Mini-batches exist for speed and noise: 64 digit images give a good-enough slope 938× cheaper per step than all 60,000, and the noise shakes weights out of shallow dips. The lab runs 15 epochs × (60000 ÷ 64 ≈ 938) ≈ 14,000 updates. Watch it: three learning rates racing on the valley, then the batch/epoch counting diagram.


In [ ]:
import numpy as np  # reuse numpy for the three races
import matplotlib.pyplot as plt  # reuse pyplot for race + counting diagram
np.random.seed(8)  # fix the seed for reproducibility
xs = np.linspace(-1, 5, 200)  # rebuild valley points
ys = (xs - 3.0) ** 2  # rebuild valley losses
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: race + counting diagram
for lr, name, col in [(0.05, 'too small (crawls)', 'green'), (0.3, 'just right', 'blue'), (1.1, 'too big (diverges)', 'red')]:  # three knobs
    w, path = 0.0, [0.0]  # start each race at w=0
    for step in range(12):  # run 12 updates per racer
        w = w - lr * 2.0 * (w - 3.0)  # one SGD step with this learning rate
        path.append(w)  # record the position
    path = np.array(path)  # array-ify for the loss math
    ax[0].plot(path, (path - 3.0) ** 2, 'o-', ms=4, color=col, label=f'lr={lr} {name}')  # draw this racer's trail
ax[0].plot(xs, ys, 'k:', alpha=0.5, label='valley')  # dotted valley underneath the racers
ax[0].set_xlabel('weight w')  # label tuned thing
ax[0].set_ylabel('loss L')  # label wrongness
ax[0].set_title('Learning rate: crawl vs converge vs explode')  # title the knob's power
ax[0].legend(fontsize=8)  # identify the three racers
N, B, E = 8, 4, 3  # toy setup: 8 samples, batch 4, 3 epochs (lab uses 60000/64/15)
ax[1].axis('off')  # diagram, not a plot
ax[1].set_xlim(0, 10)  # fix placement range
ax[1].set_ylim(0, 10)  # fix placement range
for e in range(E):  # draw one row per epoch
    ax[1].text(0.2, 8 - e * 3, f'epoch {e + 1}:', va='center', fontsize=9)  # row label
    for i in range(N):  # draw one square per sample
        ax[1].add_patch(plt.Rectangle((2 + i * 0.9, 7.3 - e * 3), 0.7, 1.2,  # sample box position
                                      facecolor='lightgreen' if i < B else 'wheat'))  # color = batch 1 vs batch 2
ax[1].text(5, 0.6, 'green = batch 1, wheat = batch 2 (2 updates per epoch)', ha='center', fontsize=9)  # legend line
ax[1].set_title('Mini-batches: 8 samples / batch 4 x 3 epochs = 6 SGD updates')  # title the counting rule
updates = E * (N // B)  # total weight updates = epochs times batches-per-epoch
print('updates = epochs x (N / batch) =', E, 'x', N // B, '=', updates)  # print the counting rule
print('lab: 15 x (60000 / 64) ~= 14000 updates')  # print the lab-scale version of the same math
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map

One training step is a loop with every § term playing a position. **Forward** (§4–§7): data enters Linear 1, bends at ReLU (§5), exits Linear 2 as logits (§6), normalizes through softmax (§7). **Score** (§8): cross-entropy grades the true class. **Backward** (§9): the `(p − onehot)/N` output error flows right-to-left — through each layer's weights (× Wᵀ) and the ReLU switch-off. **Update** (§10): SGD steps every weight downhill on one mini-batch, ~14,000 times. The map below pins each term to its station; the numbers below walk one toy step through all of them.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('data\nx', 1), ('Linear 1\n§4', 3), ('ReLU\n§5', 5), ('Linear 2\nlogits §6', 7), ('softmax\n§7', 9), ('CE loss\n§8', 11)]  # forward stations
for name, xc in stages:  # draw each forward station
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = forward pass
for xa, xb in [(1.6, 2.3), (3.7, 4.3), (5.6, 6.1), (7.9, 8.4), (9.6, 10.4)]:  # gaps between stations
    ax.annotate('', xy=(xb, 2.5), xytext=(xa, 2.5), arrowprops=dict(arrowstyle='->'))  # forward arrows
ax.text(6, 3.6, 'FORWARD: predict (§4-§7)', ha='center', fontsize=9, fontweight='bold')  # forward phase banner
ax.annotate('', xy=(1, 1.2), xytext=(11, 1.2), arrowprops=dict(arrowstyle='->', color='red', lw=2))  # long red back-arrow
ax.text(6, 0.6, 'BACKWARD (§9: blame flows back)  +  UPDATE (§10: w -= lr x grad on one mini-batch)', ha='center',  # phases banner
        fontsize=9, fontweight='bold', color='darkred')  # red = backward/update phases
ax.set_title('System map: one SGD step through every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the end-to-end numeric trace
np.random.seed(9)  # fix the seed (arrays below are hardcoded; habit kept)
X = np.array([[1.0, 0.5], [-0.5, 1.0]])  # 2 samples, 2 features (mini-batch N=2, §10)
y = np.array([0, 1])  # true classes: sample 0 -> class 0, sample 1 -> class 1
W1 = np.array([[0.5, -0.2, 0.1], [0.3, 0.4, -0.5]])  # Linear 1 weights (2 -> 3)
b1 = np.zeros(3)  # Linear 1 bias (start at zero for clarity)
W2 = np.array([[0.4, -0.3], [0.2, 0.5], [-0.1, 0.6]])  # Linear 2 weights (3 -> 2)
b2 = np.zeros(2)  # Linear 2 bias
Z1 = X @ W1 + b1  # FORWARD 1: linear step (§4)
A1 = np.maximum(0, Z1)  # FORWARD 2: bend (§5)
logits = A1 @ W2 + b2  # FORWARD 3: raw scores (§6)
shifted = logits - logits.max(axis=1, keepdims=True)  # softmax safety shift (§7)
probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)  # probabilities (§7)
loss = -np.log(probs[np.arange(2), y]).mean()  # SCORE: mean cross-entropy (§8)
dlogits = probs.copy()  # BACKWARD start: copy the prediction, then subtract the answer (§9)
dlogits[np.arange(2), y] -= 1.0  # subtract one-hot answers (§9)
dlogits /= 2.0  # average over the mini-batch (§9 + §10)
print('probs:\n', np.round(probs, 3))  # print what the net currently believes
print('loss:', round(float(loss), 4))  # print the single wrongness number
print('dL/dlogits:\n', np.round(dlogits, 3))  # print the blame seed: - on truth, + elsewhere
print('shapes: X', X.shape, '| A1', A1.shape, '| dlogits', dlogits.shape)  # batch dim rides along everywhere
dW2 = A1.T @ dlogits  # (3 x Batch) @ (Batch x 2) = (3 x 2): the §9 transpose box, aligned with W2
print('dW2 shape', dW2.shape, '== W2 shape', W2.shape)  # the transpose earns its keep: shapes align
W2_new = W2 - 0.1 * dW2  # UPDATE: one SGD step on W2 (lr=0.1, §10)
print('W2[0] before/after:', np.round(W2[0], 4), '->', np.round(W2_new[0], 4))  # print one weight's downhill step


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, on real digit images:

1. **Cell 1 (goal markdown)** — states the mission: forward, loss, backprop, SGD by hand. You already know the plot of the story (§§9-§10, mapped in §11 below).
2. **Cell 2 (setup)** — seeds plus `os.makedirs('./results')`. Same `np.random.seed(42)` habit as this lecture.
3. **Cell 3 (`load_data`)** — loads MNIST digit images (or a small built-in digits set offline) and scales pixels to [0, 1]. This is the `x` box on the left of the map.
4. **Cell 4 (`relu`, `softmax`, `init_params`)** — the building blocks: the §5 mask rule, the §7 stability shift, and fresh `W`/`b` per §4.
5. **Cell 5 (`forward`, `loss_and_grad_logits`)** — the trace scaled up: cache (X, Z1, A1, probs), then `dL/dlogits = (p − onehot)/N` exactly as in §9.
6. **Cell 6 (`backward`, `accuracy`)** — chain rule through Linear 2 (errors × Wᵀ) → §5 switch-off → Linear 1; check grad shapes match param shapes.
7. **Cell 7 (training loop)** — the §10 loop: shuffle, slice mini-batches, forward → loss → backward → `params[k] -= LR * grads[k]`, 15 epochs ≈ 14,000 updates.
8. **Cell 8 (evaluate + save)** — final train/test accuracy, `results/metrics.csv`, and the loss/accuracy curves.
9. **Cell 9 (README prompts)** — the one-liner to memorize: softmax Jacobian times CE gradient collapses to `(softmax − onehot)/N` (§9).


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Where does the gradient actually come from?**

*Answer: from the chain rule worked backwards (§9). The softmax + cross-entropy pair simplifies to* `dL/dlogits = (probabilities − one-hot labels) / batch size` *— everything before that is chain rule through each layer's weights plus the ReLU switch-off. Trace it in the §11 system-map numbers below.*

2. **What does the learning rate do, and what happens if it is far too big?**

*Answer: it scales each downhill step* `w -= lr * grad` *(§10). Too big overshoots the valley floor and the loss explodes — the red trail in the §10 race; the lab uses 0.1 and suggests trying 1.0 to watch it diverge.*

3. **Why subtract the row max before `exp` in softmax?**

*Answer: pure overflow safety (§7). It cancels out in the division so the probabilities are identical, but without it `exp(1000)` is `inf` and the whole distribution becomes `NaN`.*
